# Teleportation through a quantum passive optical network, in qg terms (§157, §158)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Viny2030/qang/blob/main/notebooks/qang_qpon_teleport.ipynb)

A downlink QPON: Alice shares cos α|00⟩ + sin α|11⟩ and sends one qubit (a photon present or absent) through a fibre (transmission T) and a passive 1:N splitter to Bob, so the photon arrives with probability η = T/N. This is the discrete-variable version of the continuous-variable study of Yaghoobianzadeh & Salehi (Academia Quantum 2026, 10.20935/AcadQuant8460). Everything runs on local simulators of IBM devices (no account, no hardware), in about 5 minutes.

## 1. Install

In [1]:
import importlib, subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "qang", "qiskit-aer", "qiskit-ibm-runtime", "scikit-learn"], check=False)
importlib.invalidate_caches()
try:
    import qang.qpon
    ok = hasattr(qang.qpon, "dual_rail_teleport_circuit")
except ImportError:
    ok = False
if not ok:  # the published qang predates these sections: install from GitHub
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                    "git+https://github.com/Viny2030/qang.git"], check=True)
    print("installed qang from GitHub; if an import fails below, restart the runtime once")
import qang
print("qang", qang.__version__)

qang 0.11.0


## 2. Set-up

In [2]:
import json, math, os, time, urllib.request
import numpy as np
import qiskit_ibm_runtime.fake_provider as fp
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

T0 = time.time()


def run(circuits, backend_name="FakeFez", shots=4000, seed=1):
    """Transpile to a fake IBM device and sample with its noise model; returns the list of counts."""
    b = getattr(fp, backend_name)()
    ts = transpile(circuits, b, optimization_level=1, seed_transpiler=137)
    res = AerSimulator.from_backend(b).run(ts, shots=shots, seed_simulator=seed).result()
    return [res.get_counts(i) for i in range(len(circuits))]


def repo_json(name):
    local = os.path.join("..", "examples", "data", name)
    if os.path.exists(local):
        return json.load(open(local, encoding="utf-8"))
    url = f"https://raw.githubusercontent.com/Viny2030/qang/main/examples/data/{name}"
    return json.loads(urllib.request.urlopen(url).read().decode("utf-8"))

from qang import qpon as Q

## 3. The optimal resource is not a Bell pair (§157)

Standard teleportation gives F = (2f + 1)/3 with singlet fraction f = (cos α + √η sin α)² / (2(cos²α + κ sin²α)). The optimum is tan α* = √η/κ, F* = (2 + η/κ)/3, and the optimal resource has qg_Z* = (κ² − η)/(κ² + η) on each qubit: one Z measurement sets the design. κ = 1 without assistance; κ = η + 1 − T when the other users report a photon that went to them (assisted protocol: a sector filter on their register).

In [3]:
print("link (L km, N)  protocol   alpha*   F*      F(pi/4)  qg_Z*")
for L, N in ((0, 4), (0, 8), (20, 4), (20, 8)):
    for prot, assisted in (("non", False), ("assisted", True)):
        eta, kap = Q.link(L, N, assisted)
        a = Q.optimal_angle(eta, kap)
        print(f"({L:2d}, {N})        {prot:9s}  {a:.3f}   {Q.optimal_fidelity(eta, kap):.4f}  "
              f"{Q.average_fidelity(math.pi / 4, eta, kap):.4f}   {Q.optimal_qg_z(eta, kap):+.3f}")

link (L km, N)  protocol   alpha*   F*      F(pi/4)  qg_Z*
( 0, 4)        non        0.464   0.7500  0.7083   +0.600
( 0, 4)        assisted   1.107   1.0000  0.9333   -0.600
( 0, 8)        non        0.340   0.7083  0.6387   +0.778
( 0, 8)        assisted   1.231   1.0000  0.8762   -0.778
(20, 4)        non        0.306   0.6998  0.6217   +0.819
(20, 4)        assisted   0.423   0.7140  0.6724   +0.663
(20, 8)        non        0.219   0.6833  0.5827   +0.905
(20, 8)        assisted   0.330   0.6921  0.6352   +0.790


On FakeFez, link (L = 0 km, N = 8): F against α for both protocols. Published (3 devices): the measured optimum sits at α*, only the optimal resource beats the classical 2/3 without assistance, and assistance wins everywhere.

In [4]:
L, N = 0, 8
T = Q.fibre_transmission(L)
alphas = [0.2, 0.4, 0.6, math.pi / 4, 1.0, 1.2, 1.4]
counts = run([Q.teleport_circuit(a, T, N, k) for a in alphas for k in range(6)])
print("alpha   F non-assisted   F assisted   kept (assisted)")
for i, a in enumerate(alphas):
    cs = counts[6 * i: 6 * i + 6]
    r0, r1 = Q.fidelity_from_counts(cs, False), Q.fidelity_from_counts(cs, True)
    print(f"{a:.3f}   {r0['fidelity']:.4f}           {r1['fidelity']:.4f}       {r1['kept']:.3f}")
for prot, assisted in (("non", False), ("assisted", True)):
    eta, kap = Q.link(L, N, assisted)
    print(f"{prot}: alpha* = {Q.optimal_angle(eta, kap):.3f}, F* = {Q.optimal_fidelity(eta, kap):.4f}")

alpha   F non-assisted   F assisted   kept (assisted)
0.200   0.6914           0.7064       0.944
0.400   0.6991           0.7565       0.850
0.600   0.6778           0.8037       0.713
0.785   0.6316           0.8485       0.565
1.000   0.5602           0.9065       0.390
1.200   0.4963           0.9417       0.252
1.400   0.4325           0.8465       0.167
non: alpha* = 0.340, F* = 0.7083
assisted: alpha* = 1.231, F* = 1.0000


## 4. The fidelity from three qg correlators, and dual rail (§158)

**A.** F = (3 + qg_XX − qg_YY + qg_ZZ)/6: three short circuits on the resource predict the teleported fidelity (published: to within 0.03 everywhere).

**B.** Dual rail: one photon in two modes per qubit, every gate conserves the photon number, and every loss empties Bob's pair, so the qg filter (weight 1 per pair) heralds all of it. Noiselessly the filtered F is 1 at any distance, with kept fraction η. Published: false heralds (a lost photon plus a device error) make the filtered F fall with distance, yet at 20 km it beats the noiseless single-rail optimum on two of three devices.

In [5]:
L, N = 0, 4
T = Q.fibre_transmission(L)
for prot, assisted in (("non", False), ("assisted", True)):
    a = Q.optimal_angle(*Q.link(L, N, assisted))
    cs = run([Q.resource_circuit(a, T, N, b) for b in "XYZ"] + [Q.teleport_circuit(a, T, N, k) for k in range(6)])
    corr = [Q.correlator_from_counts(c, assisted)[0] for c in cs[:3]]
    print(f"{prot:9s}: F from correlators {Q.fidelity_from_correlators(*corr):.4f}, "
          f"teleported {Q.fidelity_from_counts(cs[3:], assisted)['fidelity']:.4f}, "
          f"F* {Q.optimal_fidelity(*Q.link(L, N, assisted)):.4f}")

print("dual rail: link   filtered F   raw F   kept   eta    single-rail F* (best protocol)")
for L, N in ((0, 4), (0, 8), (20, 4), (20, 8)):
    eta = Q.fibre_transmission(L) / N
    r = Q.dual_rail_fidelity_from_counts(run([Q.dual_rail_teleport_circuit(math.pi / 4, eta, k) for k in range(6)]))
    best = max(Q.optimal_fidelity(*Q.link(L, N, a)) for a in (False, True))
    print(f"           ({L:2d},{N})   {r['filtered']:.4f}       {r['raw']:.4f}  {r['kept']:.3f}  {eta:.3f}  {best:.4f}")
print(f"total time {time.time() - T0:.0f} s")

non      : F from correlators 0.7437, teleported 0.7372, F* 0.7500


assisted : F from correlators 0.9701, teleported 0.9573, F* 1.0000
dual rail: link   filtered F   raw F   kept   eta    single-rail F* (best protocol)


           ( 0,4)   0.8709       0.5934  0.242  0.250  1.0000


           ( 0,8)   0.7968       0.5451  0.146  0.125  1.0000


           (20,4)   0.7790       0.5361  0.125  0.100  0.7140


           (20,8)   0.7117       0.5187  0.088  0.050  0.6921
total time 143 s


## Summary

* The design rule of a QPON resource is one qg value: qg_Z* = (κ² − η)/(κ² + η). Without assistance the Bell pair can fall below the classical limit where the optimal resource does not.
* Assistance by the other users is a sector filter.
* The teleported fidelity is a linear qg read-out of the resource.
* Dual rail turns every loss into a heralded event; the filter keeps the fidelity high and pays in rate, until false heralds set a floor.

Details: `RESEARCH_NOTES.md` §157–§158, `examples/qpon_teleport_qg.py`, `examples/qpon_teleport_dual_rail_qg.py`.